# Anisotropic Boundary Sources with Delta-Forward Scattering

![Geometry, mesh, material data, and boundary-source directions for the anisotropic boundary-source problem](images/anisotropic_boundary_problem.png)

*Problem geometry and boundary conditions. The angular panel shows a two-dimensional slice through the incident cone; the other three boundaries are vacuum.*

This tutorial applies the delta-forward scattering model to a 40 cm by 40 cm, one-group problem driven by a directional boundary source. It compares Standard P4, Standard P5, Standard P6, Galerkin One, and Galerkin Three angular operators against an equivalent pure-absorber reference.

In [ ]:
from pathlib import Path

from mpi4py import MPI

from pyopensn.aquad import GLCProductQuadrature2DXY
from pyopensn.context import Finalize, UseColor
from pyopensn.math import AngularFluxFunction
from pyopensn.mesh import OrthogonalMeshGenerator
from pyopensn.post import VolumePostprocessor
from pyopensn.solver import DiscreteOrdinatesProblem, SteadyStateSourceSolver
from pyopensn.xs import MultiGroupXS

UseColor(False)
comm = MPI.COMM_WORLD
rank = comm.rank

## Define the transport problem

The square contains one material with $\Sigma_t=1.5$ $\text{cm}^{-1}$ and $\Sigma_{s,\ell}=1.0$ $\text{cm}^{-1}$ for every retained moment. Equal scattering moments represent the delta-forward kernel, for which scattering preserves direction. It is therefore equivalent to a pure absorber with $\Sigma_a=\Sigma_t-\Sigma_s=0.5$ $\text{cm}^{-1}$. We solve that pure-absorber problem as the reference.

All solves use the same 16-direction product quadrature. The delta-forward cross-section file supplies scattering moments through P6. The three standard cases retain every available two-dimensional moment through $L=4$, $L=5$, or $L=6$, whereas Galerkin One and Galerkin Three each select a square set of 16 independent moments from the P6 expansion.

In [ ]:
domain_length = 40.0
num_cells = 40
sigma_t = 1.5
sigma_s = 1.0
sigma_a = sigma_t - sigma_s
maximum_order = 6
mu_cutoff = 0.75

nodes = [domain_length * i / num_cells for i in range(num_cells + 1)]
mesh = OrthogonalMeshGenerator(node_sets=[nodes, nodes]).Execute()
mesh.SetUniformBlockID(0)
mesh.SetOrthogonalBoundaries()

xs_filename = Path("anisotropic_boundary_delta_forward.xs")
if rank == 0:
    with xs_filename.open("w") as stream:
        stream.write("NUM_GROUPS 1\n")
        stream.write(f"NUM_MOMENTS {maximum_order + 1}\n\n")
        stream.write("SIGMA_T_BEGIN\n")
        stream.write(f"0 {sigma_t}\n")
        stream.write("SIGMA_T_END\n\n")
        stream.write("TRANSFER_MOMENTS_BEGIN\n")
        for ell in range(maximum_order + 1):
            stream.write(f"M_GFROM_GTO_VAL {ell} 0 0 {sigma_s}\n")
        stream.write("TRANSFER_MOMENTS_END\n")
comm.Barrier()

## Select and normalize the boundary directions

On the left boundary, $\mu=\boldsymbol\Omega\cdot\hat{\mathbf x}=\Omega_x$. The arbitrary boundary source is nonzero only for ordinates satisfying $\mu\geq0.75$, corresponding to a cone about the inward $+x$ normal. Its angular-flux magnitude is normalized with the discrete weights so that

$$40\,\psi_b\sum_{\Omega_{n,x}\geq0.75}w_n\Omega_{n,x}=1\ \text{particle/s}$$

per unit depth. The other three boundaries are vacuum.

In [ ]:
boundary_quadrature = GLCProductQuadrature2DXY(
    n_polar=4, n_azimuthal=8, scattering_order=maximum_order
)
selected_directions = {
    index
    for index, omega in enumerate(boundary_quadrature.omegas)
    if omega.x >= mu_cutoff
}
if not selected_directions:
    raise RuntimeError("The quadrature has no directions with mu >= 0.75.")

incoming_current = sum(
    boundary_quadrature.weights[index]
    * boundary_quadrature.omegas[index].x
    for index in selected_directions
)
boundary_strength = 1.0 / (domain_length * incoming_current)
source_rate = domain_length * boundary_strength * incoming_current

def boundary_flux(group_index, direction_index):
    if group_index == 0 and direction_index in selected_directions:
        return boundary_strength
    return 0.0

xmin_boundary = AngularFluxFunction(boundary_flux)
if rank == 0:
    print(f"BOUNDARY_DIRECTION_COUNT={len(selected_directions)}")
    print(f"BOUNDARY_SOURCE_RATE={source_rate:.12e}")

## Solve the reference and operator cases

We first solve the equivalent pure-absorber problem. We then use the delta-forward cross sections with Standard P4, Standard P5, Standard P6, Galerkin One, and Galerkin Three operators. All six calculations use the same mesh, directions, boundary source, and solver settings, so their errors are measured against a meaningful physical reference.

In [ ]:
def solve(
    operator_method, use_delta_forward=True, scattering_order=maximum_order
):
    cross_sections = MultiGroupXS()
    if use_delta_forward:
        cross_sections.LoadFromOpenSn(str(xs_filename))
    else:
        cross_sections.CreateSimpleOneGroup(sigma_t=sigma_a, c=0.0)
    quadrature = GLCProductQuadrature2DXY(
        n_polar=4,
        n_azimuthal=8,
        scattering_order=scattering_order if use_delta_forward else 0,
        operator_method=operator_method,
    )
    groupset = {
        "groups_from_to": (0, 0),
        "angular_quadrature": quadrature,
        "inner_linear_method": "petsc_gmres",
        "l_abs_tol": 1.0e-10,
        "l_max_its": 200,
        "gmres_restart_interval": 30,
    }

    problem = DiscreteOrdinatesProblem(
        mesh=mesh,
        num_groups=1,
        groupsets=[groupset],
        xs_map=[{"block_ids": [0], "xs": cross_sections}],
        boundary_conditions=[
            {"name": "xmin", "type": "arbitrary",
             "function": xmin_boundary},
            {"name": "xmax", "type": "vacuum"},
            {"name": "ymin", "type": "vacuum"},
            {"name": "ymax", "type": "vacuum"},
        ],
        options={"verbose_inner_iterations": False},
    )
    solver = SteadyStateSourceSolver(problem=problem)
    solver.Initialize()
    solver.Execute()

    total_flux = VolumePostprocessor(problem=problem, value_type="integral")
    total_flux.Execute()
    return float(total_flux.GetValue()[0][0]), problem

In [ ]:
reference_flux, _ = solve("standard", use_delta_forward=False)
standard_p4_flux, _ = solve("standard", scattering_order=4)
standard_p5_flux, _ = solve("standard", scattering_order=5)
standard_flux, _ = solve("standard")
galerkin_one_flux, galerkin_one_problem = solve("galerkin_one")
galerkin_three_flux, _ = solve("galerkin_three")
standard_p4_relative_error = (
    abs(standard_p4_flux - reference_flux) / reference_flux
)
standard_p5_relative_error = (
    abs(standard_p5_flux - reference_flux) / reference_flux
)
standard_relative_error = abs(standard_flux - reference_flux) / reference_flux
galerkin_one_relative_error = (
    abs(galerkin_one_flux - reference_flux) / reference_flux
)
galerkin_three_relative_error = (
    abs(galerkin_three_flux - reference_flux) / reference_flux
)

if rank == 0:
    print(f"PURE_ABSORBER_TOTAL_FLUX={reference_flux:.12e}")
    print(f"STANDARD_P4_TOTAL_FLUX={standard_p4_flux:.12e}")
    print(f"STANDARD_P5_TOTAL_FLUX={standard_p5_flux:.12e}")
    print(f"STANDARD_TOTAL_FLUX={standard_flux:.12e}")
    print(f"GALERKIN_ONE_TOTAL_FLUX={galerkin_one_flux:.12e}")
    print(f"GALERKIN_THREE_TOTAL_FLUX={galerkin_three_flux:.12e}")
    print(f"STANDARD_P4_RELATIVE_ERROR={standard_p4_relative_error:.12e}")
    print(f"STANDARD_P5_RELATIVE_ERROR={standard_p5_relative_error:.12e}")
    print(f"STANDARD_RELATIVE_ERROR={standard_relative_error:.12e}")
    print(f"GALERKIN_ONE_RELATIVE_ERROR={galerkin_one_relative_error:.12e}")
    print(f"GALERKIN_THREE_RELATIVE_ERROR={galerkin_three_relative_error:.12e}")

## Interpret the result

The reported response is the domain-integrated scalar flux, $\int_V\phi(\mathbf r)\,dV$, per unit depth.

| Case | Domain-integrated flux | Relative error |
|---|---:|---:|
| Pure-absorber reference | 1.964011214270 | -- |
| Standard P4 | 1.955917056047 | $4.12\times10^{-3}$ |
| Standard P5 | 1.683260656610 | $1.43\times10^{-1}$ |
| Standard P6 | 1.135792999080 | $4.22\times10^{-1}$ |
| Galerkin One | 1.964011214178 | $4.67\times10^{-11}$ |
| Galerkin Three | 1.964011214178 | $4.67\times10^{-11}$ |

Both Galerkin methods reproduce the pure-absorber reference to the solver tolerance. They select 16 independent moments for the 16 directions and construct the angular transforms as inverses. Because every selected scattering moment equals $\Sigma_s$, the resulting discrete scattering operation preserves each ordinate, as delta-forward scattering should.

Standard P4, P5, and P6 retain 15, 21, and 28 moments, respectively, for 16 directions. P4 fits within the available angular degrees of freedom and is within 0.412% of the reference. The complete P5 and P6 moment spaces are overcomplete for this quadrature; their weighted projections alias harmonics and no longer preserve each ordinate, increasing the errors to 14.3% and 42.2%. With a unit incident rate, global balance is $1=\Sigma_a\int_V\phi\,dV+L$, where $L$ is leakage. The Galerkin, Standard P4, Standard P5, and Standard P6 solutions imply approximately 1.8%, 2.2%, 15.8%, and 43.2% leakage, respectively.

## Export and visualize the scalar flux

The Galerkin One result is used as the representative scalar-flux field. The following code was run once to export it. It remains in Markdown so regression tests do not recreate the VTK files.

```python
from pyopensn.fieldfunc import FieldFunctionGridBased

scalar_flux = galerkin_one_problem.GetScalarFluxFieldFunction()[0]
FieldFunctionGridBased.ExportMultipleToPVTU(
    [scalar_flux], "Flux/AnisotropicBoundaryGalerkinOne"
)
```

![Scalar flux for the anisotropic-boundary problem](images/anisotropic.png)

In [ ]:
if rank == 0:
    xs_filename.unlink(missing_ok=True)

## Finalize (for Jupyter Notebook only)

In script mode, PyOpenSn handles finalization automatically. In a Jupyter kernel, finalize OpenSn before MPI.

In [ ]:
if "opensn_console" not in globals():
    from IPython import get_ipython

    if get_ipython() is not None:
        Finalize()
        MPI.Finalize()